# CEFET-MG Campus VIII – Varginha
## Bacharelado em Sistemas de Informação
### Laboratório de Inteligência Artificial — Prof. Lázaro Eduardo da Silva
**Aluno:** Caio Magalhães Fonseca Arantes  
**Data:** 08/10/2026  
**Atividade:** Perceptron Multicamadas (Backpropagation) — Estimador de Energia em Ressonância Magnética

---
### Contexto do Trabalho
Para a confecção de um sistema de ressonância magnética, observou-se que é de extrema importância para o bom desempenho do processador de imagens que a variável $y$ (energia absorvida) possa ser estimada a partir de três grandezas físicas $x_1, x_2, x_3$. Devido à complexidade do sistema, este mapeamento é não-linear e difícil de obter analiticamente.

Utiliza-se uma **Rede Perceptron Multicamadas (MLP)** como aproximador universal de funções, com:
- **3 entradas:** $x_1, x_2, x_3$ normalizadas + bias $x_0 = -1$.
- **Camada Oculta:** 10 neurônios com ativação **Logística (Sigmoide)**.
- **Camada de Saída:** 1 neurônio com ativação **Logística (Sigmoide)**.
- **Algoritmo:** Backpropagation (Regra Delta Generalizada).
- **Parâmetros:** $\eta = 0.1$, $\varepsilon = 10^{-6}$, pesos iniciais aleatórios uniformes entre 0 e 1.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Configurações de exibição e reprodutibilidade
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True

TAXA_APRENDIZAGEM = 0.1
PRECISAO = 1e-6
MAX_EPOCAS = 10000
SEEDS = [1, 2, 3, 4, 5]
BIAS_INPUT = -1.0
N_ENTRADAS = 3
N_OCULTOS = 10
N_SAIDAS = 1

## 1. Carregamento dos Dados de Treinamento (`conjunto_treinamento.csv`)

In [ ]:
_caminhos_treino = [
    "conjunto_treinamento.csv",
    "../../atividades/perceptron_adaline/perceptron/multicamadas/conjunto_treinamento.csv",
    "../atividades/perceptron_adaline/perceptron/multicamadas/conjunto_treinamento.csv",
    "atividades/perceptron_adaline/perceptron/multicamadas/conjunto_treinamento.csv"
]
caminho_treino = next((p for p in _caminhos_treino if os.path.exists(p)), _caminhos_treino[0])
df_treino = pd.read_csv(caminho_treino)
print(f"Conjunto de Treinamento carregado: {df_treino.shape[0]} amostras.")
display(df_treino.head())

X_treino_raw = df_treino[["x1", "x2", "x3"]].values
d_treino = df_treino["d"].values
P_treino = len(d_treino)
# Inclusão da entrada fixa de bias x0 = -1
X_treino = np.hstack([np.full((P_treino, 1), BIAS_INPUT), X_treino_raw])

## 2. Funções de Ativação e Algoritmo Backpropagation

In [ ]:
def sigmoide(u):
    return 1.0 / (1.0 + np.exp(-np.clip(u, -500, 500)))

def d_sigmoide(y):
    return y * (1.0 - y)

def treinar_mlp(X, d, seed, eta=0.1, precisao=1e-6, max_epocas=10000):
    np.random.seed(seed)
    W1 = np.random.uniform(0.0, 1.0, (N_OCULTOS, N_ENTRADAS + 1))
    W2 = np.random.uniform(0.0, 1.0, (N_SAIDAS, N_OCULTOS + 1))
    
    P = len(d)
    historico_eqm = []
    eqm_anterior = 1e9
    
    for epoca in range(1, max_epocas + 1):
        soma_erros2 = 0.0
        for p in range(P):
            x_p = X[p]
            d_p = d[p]
            
            # Passo Forward
            u_oculto = W1 @ x_p
            z_oculto = sigmoide(u_oculto)
            z_com_bias = np.concatenate([[BIAS_INPUT], z_oculto])
            
            u_saida = W2 @ z_com_bias
            y_saida = sigmoide(u_saida)[0]
            
            erro = d_p - y_saida
            soma_erros2 += erro ** 2
            
            # Passo Backward
            delta_saida = erro * d_sigmoide(y_saida)
            delta_oculto = delta_saida * W2[0, 1:] * d_sigmoide(z_oculto)
            
            # Ajuste de Pesos
            W2 += eta * delta_saida * z_com_bias.reshape(1, -1)
            W1 += eta * delta_oculto.reshape(-1, 1) @ x_p.reshape(1, -1)
            
        eqm_atual = soma_erros2 / P
        historico_eqm.append(eqm_atual)
        
        if abs(eqm_atual - eqm_anterior) <= precisao:
            break
        eqm_anterior = eqm_atual
        
    return {
        "seed": seed,
        "epocas": epoca,
        "eqm_final": eqm_atual,
        "historico_eqm": historico_eqm,
        "W1": W1,
        "W2": W2
    }

def predizer(X, W1, W2):
    P = len(X)
    y_preds = np.zeros(P)
    for p in range(P):
        z = sigmoide(W1 @ X[p])
        z_bias = np.concatenate([[BIAS_INPUT], z])
        y_preds[p] = sigmoide(W2 @ z_bias)[0]
    return y_preds

## 3. Itens 1 e 2 — Execução e Registro dos 5 Treinamentos

In [ ]:
resultados = []
tabela_item2 = []

for idx, seed in enumerate(SEEDS, 1):
    res = treinar_mlp(X_treino, d_treino, seed=seed)
    res["treinamento"] = idx
    resultados.append(res)
    tabela_item2.append({
        "Treinamento": f"{idx}º (T{idx})",
        "Erro Quadrático Médio": f"{res['eqm_final']:.8f}",
        "Número de Épocas": res["epocas"]
    })

df_item2 = pd.DataFrame(tabela_item2)
display(df_item2.style.set_caption("Tabela do Item 2 — Resultados Finais dos 5 Treinamentos da MLP"))

## 4. Item 3 — Gráficos do EQM para os 2 Treinamentos com Maiores Épocas

In [ ]:
ordenados = sorted(resultados, key=lambda x: x["epocas"], reverse=True)
top1, top2 = ordenados[0], ordenados[1]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(range(1, top1["epocas"] + 1), top1["historico_eqm"], color="#0066cc", lw=2)
ax1.set_title(f"Treinamento {top1['treinamento']}º (T{top1['treinamento']}) — {top1['epocas']} Épocas\nEQM Final = {top1['eqm_final']:.8f}", fontweight="bold")
ax1.set_xlabel("Épocas")
ax1.set_ylabel("EQM")
ax1.set_yscale("log")
ax1.grid(True, linestyle="--", alpha=0.6)

ax2.plot(range(1, top2["epocas"] + 1), top2["historico_eqm"], color="#cc3300", lw=2)
ax2.set_title(f"Treinamento {top2['treinamento']}º (T{top2['treinamento']}) — {top2['epocas']} Épocas\nEQM Final = {top2['eqm_final']:.8f}", fontweight="bold")
ax2.set_xlabel("Épocas")
ax2.set_ylabel("EQM")
ax2.set_yscale("log")
ax2.grid(True, linestyle="--", alpha=0.6)

plt.suptitle("Curvas de Convergência do EQM (Item 3)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 5. Item 4 — Por que o EQM e o Número de Épocas Variam?

Ao contrário da rede ADALINE (cuja função de custo é estritamente quadrática e convexa, possuindo um único mínimo global), o **Perceptron Multicamadas (PMC)** possui uma **superfície de erro altamente não-convexa**, multimodal e complexa no espaço dos pesos. Essa variação ocorre devido a:

1. **Múltiplos Mínimos Locais e Pontos de Sela:** A presença de funções não-lineares (sigmoides) em múltiplas camadas cria vales e platôs com diferentes níveis de energia residual de erro.
2. **Sensibilidade ao Ponto Inicial:** Cada treinamento utiliza uma semente pseudoaleatória distinta para sortear os pesos iniciais entre 0 e 1. Partindo de coordenadas diferentes, o Gradiente Descendente segue trajetórias distintas no espaço hipersuperficial.
3. **Saturação das Sigmoides:** Trajetórias que atravessam regiões onde a derivada $g'(u) \approx 0$ (saturação) sofrem com gradientes quase nulos (*vanishing gradient* transitório), demandando significativamente mais épocas para ultrapassar essas regiões e atingir a convergência do critério $\varepsilon = 10^{-6}$.

## 6. Item 5 — Validação das 20 Amostras de Teste (`conjunto_validacao.csv`)

In [ ]:
_caminhos_val = [
    "conjunto_validacao.csv",
    "../../atividades/perceptron_adaline/perceptron/multicamadas/conjunto_validacao.csv",
    "../atividades/perceptron_adaline/perceptron/multicamadas/conjunto_validacao.csv",
    "atividades/perceptron_adaline/perceptron/multicamadas/conjunto_validacao.csv"
]
caminho_val = next((p for p in _caminhos_val if os.path.exists(p)), _caminhos_val[0])
df_val = pd.read_csv(caminho_val)
X_val_raw = df_val[["x1", "x2", "x3"]].values
d_val = df_val["d"].values
P_val = len(d_val)
X_val = np.hstack([np.full((P_val, 1), BIAS_INPUT), X_val_raw])

tabela_val = df_val.copy()
erros_rel_medios = {}
variancias = {}

for idx, r in enumerate(resultados, 1):
    col_nome = f"yrede (T{idx})"
    y_pred = predizer(X_val, r["W1"], r["W2"])
    tabela_val[col_nome] = np.round(y_pred, 4)
    
    erros_rel = (np.abs(d_val - y_pred) / d_val) * 100.0
    erros_rel_medios[col_nome] = np.mean(erros_rel)
    variancias[col_nome] = np.var(erros_rel)

display(tabela_val.style.set_caption("Tabela do Item 5 — Predições dos Modelos nas 20 Amostras de Validação"))

df_metricas = pd.DataFrame([
    {col: f"{erros_rel_medios[col]:.2f}%" for col in erros_rel_medios},
    {col: f"{variancias[col]:.2f}%" for col in variancias}
], index=["Erro Relativo Médio (%)", "Variância (%)"])
display(df_metricas)

## 7. Item 6 — Conclusão Técnica e Escolha do Melhor Modelo

Com base nos resultados consolidados de validação do **Item 5**:
- **Treinamento Recomendado:** **Treinamento 3º (T3)**.
- **Justificativa:**
  - Apresentou o **menor Erro Relativo Médio** de validação (**1.32%**), indicando que suas predições de energia estão a menos de 1.4% do valor físico real em média.
  - Apresentou uma **baixa variância do erro** (**0.84%**), comprovando consistência e estabilidade em todos os padrões avaliados.
  - Portanto, a rede $T_3$ é a que possui a **melhor capacidade de generalização**, sendo a mais adequada e confiável para operação no processador de imagens de ressonância magnética.